# FlashNystrom Quickstart

Build and try the package end to end. The notebook clones the repo, builds the CUDA extension against the runtime GPU, runs a smoke test, and walks a latency sweep up to context length N=524288 (512K tokens).

**GPU requirement.** Compute capability 8.0 or newer (Ampere, Ada, Hopper, Blackwell). Free-tier T4 (sm_75) is *not* supported; switch the Colab runtime to L4 or A100 from `Runtime` → `Change runtime type`. The build step prints a clear error if it detects an older GPU.

**Wall-clock.** Build is ~3-5 minutes on Colab. Smoke test runs in seconds. The latency sweep to N=512K takes another ~5-10 minutes on A100 (most of that is SDPA itself — by 256K it is taking seconds per call, by 512K tens of seconds; FN is microseconds either way). Total: ~10-15 minutes.

## 1. Confirm the GPU

Build fails on sm_75 and earlier. If this prints `compute_cap=(7, 5)` or older, switch the runtime before continuing.

In [ ]:
import torch
assert torch.cuda.is_available(), "No CUDA GPU detected. Set Runtime -> Change runtime type to GPU."
cap = torch.cuda.get_device_capability(0)
print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"compute cap: {cap}")
print(f"torch:       {torch.__version__}")
assert cap[0] >= 8, (
    f"FlashNystrom requires compute capability 8.0+ (Ampere/Ada/Hopper/Blackwell). "
    f"This runtime has sm_{cap[0]}{cap[1]}, which is not supported. "
    f"Switch to L4 or A100 in Runtime -> Change runtime type."
)

## 2. Clone and build

Clones with `--recursive` to pull the CUTLASS submodule. `--no-build-isolation` reuses the Colab Python env (which already has torch + CUDA), so we don't pay for a fresh torch install in the build sandbox.

In [ ]:
!git clone --recursive https://github.com/athrva98/FlashNystrom.git
%cd FlashNystrom
!pip install -q ninja pytest
!pip install -e . --no-build-isolation

## 3. Smoke test

Forward and backward through `flash_nystrom_attention` on a small input. Catches the build-but-doesn't-run case (extension imports but kernels fail on this GPU).

In [ ]:
from flash_nystrom import flash_nystrom_attention

B, H, N, D = 1, 4, 2048, 64
q = torch.randn(B, H, N, D, dtype=torch.float16, device="cuda")
k = torch.randn(B, H, N, D, dtype=torch.float16, device="cuda")
v = torch.randn(B, H, N, D, dtype=torch.float16, device="cuda", requires_grad=True)

out = flash_nystrom_attention(q, k, v, num_landmarks=32, newton_iter=6)
out.sum().backward()

assert out.shape == (B, H, N, D)
assert v.grad is not None and torch.isfinite(v.grad).all()
print(f"smoke test ok: out.shape={tuple(out.shape)}, v.grad finite")

## 4. Run the test suite

Optional. 84 tests, well under a minute on A100. Skip if you just want to use the kernels and not verify them.

In [ ]:
!pytest tests/ -q --tb=short

## 5. Latency demo

Latency sweep from N=512 up to N=524288 (512K tokens). FN's per-call cost scales linearly with N; SDPA's scales quadratically and becomes the dominant wall-clock cost above ~64K. The crossover between the two is around N=2048.

Repetition counts are tiered: full repeats at small N, fewer at large N so the sweep finishes in reasonable wall-clock time. The reported number is the median of those repeats.

Set `MAX_N` below to a smaller value (e.g. 65536) for a quicker run.

In [ ]:
import torch
import torch.nn.functional as F
from flash_nystrom import flash_nystrom_attention

MAX_N = 524288  # 512K tokens. Reduce to 65536 for a fast pass.

def reps_for(N):
    """Tiered (warmup, repeat). SDPA at very large N costs seconds per call,
    so dial the reps down to keep total wall-clock reasonable."""
    if N <= 8192:   return 5, 20
    if N <= 32768:  return 3, 10
    if N <= 131072: return 2, 5
    return 1, 3

def time_call(fn, warmup, repeat):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    events = [(torch.cuda.Event(enable_timing=True),
               torch.cuda.Event(enable_timing=True)) for _ in range(repeat)]
    for s, e in events:
        s.record(); fn(); e.record()
    torch.cuda.synchronize()
    return sorted(s.elapsed_time(e) for s, e in events)[repeat // 2]

B, H, D, m, niter = 1, 4, 64, 32, 6
seq_lengths = [N for N in [512, 1024, 2048, 4096, 8192, 16384,
                            32768, 65536, 131072, 262144, 524288] if N <= MAX_N]

print(f"{'N':>7} | {'FN fwd (ms)':>12} | {'SDPA fwd (ms)':>14} | {'SDPA / FN':>10}")
print("-" * 54)
rows = []
for N in seq_lengths:
    warmup, repeat = reps_for(N)
    q = torch.randn(B, H, N, D, dtype=torch.float16, device="cuda")
    k = torch.randn(B, H, N, D, dtype=torch.float16, device="cuda")
    v = torch.randn(B, H, N, D, dtype=torch.float16, device="cuda")

    fn_t = time_call(lambda: flash_nystrom_attention(q, k, v, num_landmarks=m, newton_iter=niter),
                     warmup=warmup, repeat=repeat)
    sd_t = time_call(lambda: F.scaled_dot_product_attention(q, k, v),
                     warmup=warmup, repeat=repeat)
    ratio = sd_t / fn_t
    rows.append((N, fn_t, sd_t, ratio))
    print(f"{N:>7} | {fn_t:>12.3f} | {sd_t:>14.3f} | {ratio:>9.2f}x")
    # Free GPU memory before the next, larger N.
    del q, k, v
    torch.cuda.empty_cache()

## 6. Module form

`FlashNystromAttention` is a regular `nn.Module`. Drop it into a transformer block in place of an SDPA call.

In [ ]:
from flash_nystrom import FlashNystromAttention, NystromConfig

cfg = NystromConfig(num_landmarks=64, newton_iter=6, conv_kernel_size=3)
attn = FlashNystromAttention(dim=512, heads=8, config=cfg).cuda()

x = torch.randn(4, 4096, 512, device="cuda", dtype=torch.float16)
y = attn(x)
y.sum().backward()
print(f"FlashNystromAttention output: {tuple(y.shape)}")
print(f"trainable parameters:        {sum(p.numel() for p in attn.parameters()):,}")

## What's next

- The full latency sweep including the fwd+bwd column and the pure-PyTorch reference: `python benchmarks/bench_fwd_bwd.py`. Walks N from 128 to 262144. Takes about 5-10 minutes on A100.
- The CIFAR-10 example trainer: `python examples/train_cifar10.py`.
- The longer technical writeup and kernel-by-kernel design discussion is in the README's "SMEM sizing and occupancy" and "Scope" sections.

Bug reports and questions: https://github.com/athrva98/FlashNystrom/issues